# 03 TypeB 上機教材：預測如何變成可交易策略？

**Big HOT：一個 55% 準確率的模型，為什麼可能賺錢，也可能賠錢？**

這堂課把 Week 3 的 `prob_up` 轉成交易訊號、部位、報酬、成本與回測診斷。核心提醒：**prediction is not P&L**。


## 課前閱讀與 LMS 回應

課前請先閱讀 [Week 4 main](../../04/week4_main.md)。

課前 HOT：

```text
A model has 55% accuracy. Is it enough to trade?
```

LMS 回應格式：

```text
My answer:
55% accuracy is / is not enough because ______.

The missing trading assumptions are:
1.
2.
3.

The first risk I would test in Colab is ______.
```

課中使用方式：開場先統計 yes / no / depends，再用 threshold、position、cost、turnover 與 drawdown 修正判斷。


## 3 小時 HOT storyline

| 時間 | Cluster | 核心 HOT | 可見產出 |
|---:|---|---|---|
| 0:00-0:25 | C1. Probability to position | 55% 準確率足夠交易嗎？ | decision rule |
| 0:25-1:05 | C1. Threshold design | threshold 提高，交易變少，是好是壞？ | threshold table |
| 1:05-1:45 | C2. Backtest correctness | 哪一行程式偷看未來？ | bias debug |
| 1:45-2:20 | C2. Costs and turnover | 成本提高後哪個策略先死？ | cost sensitivity |
| 2:20-2:50 | C3. Robustness | Sharpe、drawdown、turnover 誰最該優先？ | ranking |
| 2:50-3:00 | C4. Handoff | 哪些假設要帶去 Week 8 風險審查？ | assumption checklist |


## Learning Loop Map（新版 Type B 操作版）

| Loop | Mini-input | BIT / HOT | Visible output | Delayed feedback focus |
|---|---|---|---|---|
| 1 | Accuracy vs P&L | Predict + justify | accuracy-to-trading judgment | accuracy 不等於獲利 |
| 2 | Signal to position | Transform | position rule statement | probability 如何變成交易行為 |
| 3 | Threshold design | Predict + test | threshold interpretation table | trade rate、turnover、Sharpe 的 trade-off |
| 4 | Backtest correctness | Debug | look-ahead bias debug note | position timing vs return timing |
| 5 | Cost and turnover | Stress test | cost sensitivity interpretation | 成本如何改變策略結論 |
| 6 | Robustness and handoff | Rank / completion | metric ranking 與 assumption checklist | Week 8 風險審查前必須揭露什麼 |


## TypeB 課堂語言

```text
A higher threshold helps because ______, but hurts because ______.
This backtest is biased because ______.
The strategy earns before cost / after cost because ______.
I would reject this strategy if ______.
```


## 學生回應方式（課中使用）

本堂課每個回答都要把 prediction 連到 P&L assumption。

| 場景 | 回應格式 |
|---|---|
| Accuracy | `Accuracy is not enough because ______.` |
| Threshold | `A higher threshold helps because ______, but hurts because ______.` |
| Backtest bug | `This line is biased because ______.` |
| Cost sensitivity | `The conclusion changes when ______.` |

小組分享時請先說自己的初始預測，再說 Colab output 是否改變了你的想法。


In [ ]:
# 課堂穩定性設定：預設使用合成 OHLCV 資料。
# 若教室網路穩定且已安裝 yfinance，可以把 USE_ONLINE_DATA 改成 True。
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

SYMBOL = "0050.TW"
USE_ONLINE_DATA = False


def make_synthetic_ohlcv(n=760, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2021-01-01", periods=n)
    t = np.arange(n)
    regime = np.select(
        [t < n * 0.33, t < n * 0.66],
        [0.00045, -0.00015],
        default=0.00025,
    )
    shocks = rng.normal(0, 0.011, n)
    shocks[0] = rng.normal(0, 0.011)
    ret = regime + shocks + 0.08 * np.r_[0, shocks[:-1]]
    close = 100 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.012, n))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.012, n))
    volume = rng.lognormal(mean=15.2, sigma=0.25, size=n) * (1 + 8 * np.abs(ret))
    df = pd.DataFrame(
        {
            "Open": open_,
            "High": high,
            "Low": low,
            "Close": close,
            "Adj Close": close,
            "Volume": volume.astype(int),
        },
        index=dates,
    )
    df.index.name = "Date"
    return df


def load_market_data(symbol=SYMBOL, start="2020-01-01", use_online=USE_ONLINE_DATA):
    if use_online:
        try:
            import yfinance as yf
            df = yf.download(symbol, start=start, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if not df.empty and {"Open", "High", "Low", "Close", "Volume"}.issubset(df.columns):
                print(f"Loaded online data: {symbol}, rows={len(df)}")
                return df.dropna()
        except Exception as exc:
            print("Online download failed; falling back to synthetic data.")
            print(type(exc).__name__, exc)
    print("Using synthetic OHLCV data. Toggle USE_ONLINE_DATA=True for real market data.")
    return make_synthetic_ohlcv()


def add_features(raw):
    df = raw.copy()
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_fwd_1d"] = df["Close"].shift(-1) / df["Close"] - 1
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_gap"] = df["ma_5"] / df["ma_20"] - 1
    df["mom_5"] = df["Close"] / df["Close"].shift(5) - 1
    df["mom_20"] = df["Close"] / df["Close"].shift(20) - 1
    df["vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    volume_mean = df["Volume"].rolling(20).mean()
    volume_std = df["Volume"].rolling(20).std()
    df["volume_z"] = (df["Volume"] - volume_mean) / volume_std
    df = df.dropna()
    df["target_up"] = (df["ret_fwd_1d"] > 0).astype(int)
    return df


def max_drawdown(ret):
    wealth = (1 + ret.fillna(0)).cumprod()
    dd = wealth / wealth.cummax() - 1
    return dd.min()


def sharpe(ret, periods=252):
    vol = ret.std()
    if vol == 0 or np.isnan(vol):
        return np.nan
    return np.sqrt(periods) * ret.mean() / vol


def perf_table(returns_dict):
    rows = []
    for name, ret in returns_dict.items():
        ret = pd.Series(ret).dropna()
        rows.append(
            {
                "strategy": name,
                "ann_return": (1 + ret).prod() ** (252 / len(ret)) - 1 if len(ret) else np.nan,
                "ann_vol": ret.std() * np.sqrt(252),
                "sharpe": sharpe(ret),
                "max_drawdown": max_drawdown(ret),
                "win_rate": (ret > 0).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("strategy").round(4)


raw = load_market_data()
df = add_features(raw)
df.tail()


## 建立一個可回測的模型機率

為了讓 notebook 可獨立執行，這裡重新訓練一個簡單模型。Week 3 若已有自己的模型輸出，也可以直接把 `prob_up` 欄位接進來。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feature_cols = ["ma_gap", "mom_5", "mom_20", "vol_20", "volume_z", "range_pct"]
split_idx = int(len(df) * 0.70)

train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

model = RandomForestClassifier(n_estimators=250, max_depth=4, random_state=11)
model.fit(train[feature_cols], train["target_up"])
test["prob_up"] = model.predict_proba(test[feature_cols])[:, 1]
test["pred_up"] = (test["prob_up"] > 0.50).astype(int)

round(accuracy_score(test["target_up"], test["pred_up"]), 4)


## C1 HOT 1：55% accuracy 是否足夠交易？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Predict + justify |
| Think | 60 秒：先判斷 55% accuracy 是否足夠交易。 |
| Pair/Group | 3 分鐘：兩人加入成本與 payoff 後修正答案。 |
| Visible output | accuracy-to-trading judgment。 |
| Delayed feedback | 追問：What about payoff size, turnover, and cost? |

HOT 類型：**Predict + Justify**  
先問學生：如果模型 accuracy 是 55%，你會交易嗎？答案必須包含成本、錯誤類型、交易頻率與報酬大小。


In [ ]:
accuracy_is_not_enough = pd.DataFrame(
    [
        ["55% accuracy, tiny average win, high cost", "probably no", "edge may disappear after costs"],
        ["52% accuracy, big wins and small losses", "maybe", "payoff asymmetry matters"],
        ["60% accuracy, trades every day", "unclear", "turnover and overfitting risk"],
        ["50% accuracy, strong risk filter", "unclear", "may avoid bad regimes"],
    ],
    columns=["case", "initial_decision", "why"],
)
accuracy_is_not_enough


## C1 HOT 2：probability 如何變成 position？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Transform |
| Think | 60 秒：個人選 long-flat / long-short / scaled。 |
| Pair/Group | 4 分鐘：小組說明 position rule 的風險。 |
| Visible output | position rule statement。 |
| Delayed feedback | 比較兩組規則，問哪個更容易治理。 |

HOT 類型：**Transform**  
同一個模型輸出，可以轉成不同部位規則。規則不同，績效與風險就不同。


In [ ]:
def make_position(prob_up, threshold=0.55, mode="long_flat"):
    if mode == "long_flat":
        return pd.Series(np.where(prob_up > threshold, 1.0, 0.0), index=prob_up.index)
    if mode == "long_short":
        return pd.Series(np.where(prob_up > threshold, 1.0, np.where(prob_up < 1 - threshold, -1.0, 0.0)), index=prob_up.index)
    if mode == "scaled":
        raw_pos = (prob_up - 0.50) * 4
        return pd.Series(np.clip(raw_pos, -1, 1), index=prob_up.index)
    raise ValueError("unknown mode")

position_rules = pd.DataFrame(
    [
        ["long_flat", "long if prob_up > threshold else cash", "simple, lower risk"],
        ["long_short", "long high prob, short low prob", "more aggressive"],
        ["scaled", "position size proportional to confidence", "smooth but harder to explain"],
    ],
    columns=["rule", "definition", "risk"],
)
position_rules


## C1 HOT 3：threshold 提高後，策略會更保守還是更危險？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Predict + test |
| Think | 60 秒：預測 threshold 提高後 trade rate、turnover、Sharpe。 |
| Pair/Group | 4 分鐘：小組對照 threshold table 修正預測。 |
| Visible output | threshold interpretation table。 |
| Delayed feedback | 延後回饋：Why can fewer trades be better or worse? |

HOT 類型：**Predict + Test**  
學生先預測 trade rate、turnover、Sharpe、drawdown 會如何改變，再跑表格驗證。


In [ ]:
def backtest_from_position(market, position, cost=0.001):
    # position at date t earns next-period return ret_fwd_1d; turnover pays cost when position changes.
    turnover = position.diff().abs().fillna(position.abs())
    gross = position * market["ret_fwd_1d"]
    net = gross - turnover * cost
    return pd.DataFrame({"position": position, "turnover": turnover, "gross_ret": gross, "net_ret": net})

threshold_rows = []
backtests = {}
for th in [0.50, 0.53, 0.55, 0.58, 0.60, 0.65]:
    pos = make_position(test["prob_up"], threshold=th, mode="long_flat")
    bt = backtest_from_position(test, pos, cost=0.001)
    backtests[th] = bt
    threshold_rows.append(
        {
            "threshold": th,
            "trade_rate": (pos != 0).mean(),
            "avg_turnover": bt["turnover"].mean(),
            "net_sharpe": sharpe(bt["net_ret"]),
            "max_drawdown": max_drawdown(bt["net_ret"]),
        }
    )
threshold_table = pd.DataFrame(threshold_rows).round(4)
threshold_table


## C2 HOT 4：哪一行程式可能偷看未來？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Debug |
| Think | 60 秒：個人找可疑回測行。 |
| Pair/Group | 4 分鐘：小組說明訊號形成時間與報酬實現時間。 |
| Visible output | look-ahead bias debug note。 |
| Delayed feedback | 追問：When was the position formed? |

HOT 類型：**Debug**  
正確邏輯：今天收盤後形成 position，賺的是下一期報酬。錯誤邏輯常把今天報酬拿來當今天已知的交易結果。


In [ ]:
chosen_th = 0.55
pos = make_position(test["prob_up"], threshold=chosen_th, mode="long_flat")

correct_bt = backtest_from_position(test, pos, cost=0.001)
bug_return = pos * test["ret_1d"]  # intentionally suspicious: same-day return may not be tradable

bias_compare = perf_table(
    {
        "correct_next_period_net": correct_bt["net_ret"],
        "suspicious_same_day_gross": bug_return,
        "buy_and_hold": test["ret_fwd_1d"],
    }
)
bias_compare


## C2 HOT 5：交易成本提高後，策略哪裡先壞掉？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Stress test |
| Think | 60 秒：預測成本提高後哪個指標先壞。 |
| Pair/Group | 4 分鐘：小組解讀 cost sensitivity。 |
| Visible output | cost sensitivity interpretation。 |
| Delayed feedback | 比較不同組結論，問哪個假設最改變決策。 |

HOT 類型：**Stress test + Explain**  
成本不是回測最後才加的小細節；成本會改變 threshold、turnover 與策略可行性。


In [ ]:
cost_rows = []
for cost in [0.0, 0.0005, 0.0010, 0.0015, 0.0020, 0.0030]:
    bt = backtest_from_position(test, pos, cost=cost)
    cost_rows.append(
        {
            "one_way_cost": cost,
            "ann_return": (1 + bt["net_ret"]).prod() ** (252 / len(bt)) - 1,
            "sharpe": sharpe(bt["net_ret"]),
            "max_drawdown": max_drawdown(bt["net_ret"]),
            "avg_turnover": bt["turnover"].mean(),
        }
    )
cost_sensitivity = pd.DataFrame(cost_rows).round(4)
cost_sensitivity


## C3 HOT 6：哪個績效指標最能說服你？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Rank/order |
| Think | 60 秒：個人排序最重要績效指標。 |
| Pair/Group | 3 分鐘：小組以研究員/風控/PM 角色比較排序。 |
| Visible output | metric ranking。 |
| Delayed feedback | 追問：Which metric would a risk manager care about first? |

HOT 類型：**Rank + Defend**  
請學生把 annual return、Sharpe、max drawdown、turnover、win rate 排序，並說明這堂課的策略應該優先看哪三個。


In [ ]:
strategies = {
    "buy_and_hold": test["ret_fwd_1d"],
    "long_flat_055": backtests[0.55]["net_ret"],
    "long_flat_060": backtests[0.60]["net_ret"],
    "long_flat_065": backtests[0.65]["net_ret"],
}
perf = perf_table(strategies)
perf["avg_turnover"] = [
    0.0,
    backtests[0.55]["turnover"].mean(),
    backtests[0.60]["turnover"].mean(),
    backtests[0.65]["turnover"].mean(),
]
perf.round(4)


In [ ]:
equity = pd.DataFrame({name: (1 + ret.fillna(0)).cumprod() for name, ret in strategies.items()})
equity.plot(title="Equity curves after cost")
plt.show()


## C3 HOT 7：策略看起來好，是因為 threshold 還是因為 market regime？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare + challenge |
| Think | 60 秒：先判斷績效是否來自單一 regime。 |
| Pair/Group | 4 分鐘：小組比較 subperiod 結果。 |
| Visible output | regime risk statement。 |
| Delayed feedback | 問：Is this evidence of failure or sample instability? |

HOT 類型：**Compare + Challenge**  
將測試期切成幾段，檢查策略是否只在某一段有效。


In [ ]:
bt055 = backtests[0.55].copy()
cut_points = np.linspace(0, len(bt055), 4, dtype=int)
subperiod_rows = []
for i in range(3):
    part = bt055.iloc[cut_points[i] : cut_points[i + 1]]
    subperiod_rows.append(
        {
            "subperiod": i + 1,
            "start": part.index.min().date(),
            "end": part.index.max().date(),
            "sharpe": sharpe(part["net_ret"]),
            "ann_return": (1 + part["net_ret"]).prod() ** (252 / len(part)) - 1,
            "max_drawdown": max_drawdown(part["net_ret"]),
        }
    )
pd.DataFrame(subperiod_rows).round(4)


## C4 HOT 8：Week 8 風險審查前，你必須揭露哪些假設？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Completion |
| Think | 60 秒：個人勾選最不能漏揭露的假設。 |
| Pair/Group | 4 分鐘：小組完成 assumption checklist。 |
| Visible output | assumption checklist。 |
| Delayed feedback | 教師最後收斂成 Week 8 的風險審查問題。 |

HOT 類型：**Checklist + Justify**  
每組要把「策略看起來不錯」改成「策略在這些假設下看起來不錯」。


In [ ]:
assumption_checklist = pd.DataFrame(
    {
        "assumption": [
            "features available before trade",
            "transaction cost estimate is realistic",
            "threshold chosen before final test",
            "benchmark is appropriate",
            "drawdown is acceptable",
            "result is not driven by one short subperiod",
            "model can be explained to a risk committee",
        ],
        "status": "",
        "evidence_or_concern": "",
    }
)
assumption_checklist


## Optional HOT：3 小時內時間不夠時可跳過

1. **Long-short compare**：long-short 是否一定比較好？
2. **Scaled position**：部位大小跟信心成比例，會更穩健嗎？
3. **Drawdown stop**：加停損會降低風險，還是破壞策略？
4. **Parameter honesty**：threshold 是事前設計還是事後挑出來的？


In [ ]:
optional_modes = {}
for mode in ["long_flat", "long_short", "scaled"]:
    pos_mode = make_position(test["prob_up"], threshold=0.55, mode=mode)
    optional_modes[mode] = backtest_from_position(test, pos_mode, cost=0.001)["net_ret"]
perf_table(optional_modes)


## Appendix HOT A1：Backtest overfitting 怎麼發生？

HOT 類型：**Simulate + Warn**  
即使沒有真訊號，試很多參數也可能找到一個看似漂亮的策略。


In [ ]:
rng = np.random.default_rng(123)
random_trials = []
for trial in range(200):
    random_pos = pd.Series(rng.choice([0, 1], size=len(test), p=[0.5, 0.5]), index=test.index)
    random_bt = backtest_from_position(test, random_pos, cost=0.001)
    random_trials.append(sharpe(random_bt["net_ret"]))

pd.Series(random_trials).plot(kind="hist", bins=30, title="Distribution of Sharpe from random trial rules")
plt.show()
pd.Series(random_trials).describe().round(4)


## Appendix HOT A2：Walk-forward 與 purged CV 各自解決什麼問題？

HOT 類型：**Compare**  
Walk-forward 強調時間前進；purged CV 強調標籤重疊與資訊污染。


In [ ]:
validation_methods = pd.DataFrame(
    [
        ["single holdout", "simple future test", "high variance if one test period"],
        ["walk-forward", "repeated future-like testing", "more work, still parameter choices"],
        ["purged CV", "reduces label overlap contamination", "harder to implement and explain"],
        ["combinatorial purged CV", "many train/test path combinations", "advanced; useful for strategy selection risk"],
    ],
    columns=["method", "helps_with", "limitation"],
)
validation_methods


## Appendix HOT A3：交易成本可以視為一種 regularization 嗎？

HOT 類型：**Bridge**  
成本懲罰高換手，效果類似限制策略不要過度頻繁調整。


In [ ]:
turnover_penalty = []
for th in np.linspace(0.50, 0.70, 9):
    pos_th = make_position(test["prob_up"], threshold=float(th), mode="long_flat")
    bt = backtest_from_position(test, pos_th, cost=0.001)
    turnover_penalty.append(
        {
            "threshold": round(float(th), 2),
            "trade_rate": (pos_th != 0).mean(),
            "avg_turnover": bt["turnover"].mean(),
            "net_sharpe": sharpe(bt["net_ret"]),
        }
    )
pd.DataFrame(turnover_penalty).round(4)


## Appendix HOT A4：情境測試比單一路徑回測多看見什麼？

HOT 類型：**Scenario design**  
問學生：如果未來是高波動盤整、快速下跌、低波動上漲，這個策略哪個情境最脆弱？


In [ ]:
scenario_table = pd.DataFrame(
    [
        ["low_vol_uptrend", "trend signals may work", "watch under-trading"],
        ["high_vol_sideways", "false signals increase", "reduce size or raise threshold"],
        ["fast_drawdown", "long-only suffers", "drawdown limit / risk overlay"],
        ["cost_spike", "turnover becomes expensive", "trade less often"],
    ],
    columns=["scenario", "expected_strategy_behavior", "possible_control"],
)
scenario_table


## Learning Evidence Checklist

本堂課結束前，至少留下這些 evidence：

- [ ] accuracy-to-trading judgment。
- [ ] position rule statement。
- [ ] threshold interpretation table。
- [ ] look-ahead bias debug note。
- [ ] cost sensitivity interpretation。
- [ ] metric ranking。
- [ ] assumption checklist for risk review。

Exit ticket：

```text
The strategy looks promising only if ______.
The assumption I must disclose is ______.
The risk review should focus on ______.
```


## 參考資料

本課程設計參考下列概念來源，重點不是要求學生讀完整篇，而是把研究中的核心判斷轉成上機問題。

- López de Prado, M. (2018). *Advances in Financial Machine Learning*. 用於 financial ML failure、labeling、triple-barrier、meta-labeling、finance cross-validation、backtest overfitting。
- Gu, S., Kelly, B., & Xiu, D. (2020). Empirical Asset Pricing via Machine Learning. *Review of Financial Studies*. 用於 momentum、liquidity、volatility、非線性模型與資產報酬預測。
- Machine Learning and Portfolio Optimization 相關文獻。用於 regularization、cross-validation、estimation error 與 portfolio construction。
- Robust perspective on transaction costs in portfolio optimization 相關技術筆記。用於 transaction cost、turnover、robustness。
- XAI in finance 綜述文獻。用於 feature importance、SHAP、trust、risk assessment、governance。
- Lee, W.-Y. momentum-based sentiment trading strategy 相關研究。用於 Appendix 中 momentum + sentiment、benchmark、transaction cost、long-horizon evaluation。
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 用於 practical ML workflow、train/test、validation、classification metrics、error analysis。
